# Synthetic Speech Data Pipeline

## mount

In [ ]:
# mount
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


## installation

### For Unsloth

In [ ]:
%%capture
# install Unsloth, xformers, sentencepiece, (for manipulation the text) - bitsandbytes (quantization) - peft finetune (adapter) - trl (trainer)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
!git clone https://github.com/SparkAudio/Spark-TTS

Cloning into 'Spark-TTS'...
remote: Enumerating objects: 384, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 384 (delta 120), reused 99 (delta 99), pack-reused 213 (from 1)
Receiving objects: 100% (384/384), 7.07 MiB | 20.69 MiB/s, done.
Resolving deltas: 100% (156/156), done.


In [ ]:
! pip install omegaconf einx torchcodec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.6 MB/s eta 0:00:00


In [ ]:
! pip install -q transformers
! pip install -q sentencepiece

### For NAMAA model (Error in running incombitaple)

In [ ]:
# ! pip install -q chatterbox-tts

### For Edge-tts

In [ ]:
!pip install -q edge-tts

## importing

In [ ]:
import pandas as pd
import re, os
from typing import Callable, Tuple

## download model

In [ ]:
# for unslouth
from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#snapshot_download("SparkAudio/Spark-TTS-0.5B", local_dir = "/content/drive/MyDrive/Olimi AI /model/Spark-TTS-0.5B")
model, tokenizer = FastModel.from_pretrained(
    model_name="/gdrive/MyDrive/Olimi AI /model/Spark-TTS-0.5B/LLM",
    max_seq_length= 2048, # max tokens
    dtype= torch.float32, # datatype used from spark
    full_finetuning=True,
    load_in_4bit= False

)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## data

In [ ]:
# data

path_sports_data = "/gdrive/MyDrive/Olimi AI /TE_Sports.xlsx"
path_Wikipedia_data = "/gdrive/MyDrive/Olimi AI /8.Arabic_Egyptian_Wikipedia.xlsx"
path_tweeets_data = "/gdrive/MyDrive/Olimi AI /2.Egyptian Tweets.xlsx"


sports_data = pd.read_excel(path_sports_data, usecols=["Text"]).squeeze()
Wikipedia_data = pd.read_excel(path_Wikipedia_data, usecols=["Text"]).squeeze()
tweeets_data = pd.read_excel(path_tweeets_data, usecols=["Text"]).squeeze()

sports_data.head(5)


,Text
0,تحية عناصر لأحمد حسن الفيديو ..
1,ق15: هجمة خطيرة للأهلي، أحمد فتحي يرسل عرضية ل...
2,بعد جاحة دي انا هسحب كلامي تاني 😂😂✋
3,ربع ساعه والتعادل السلبي
4,فرصة في منتهي الخطورة من عبد الله و لكن الدفاع...


In [ ]:
Wikipedia_data.head(5)

,Text
0,ابريل\nابريل هو اليوم فى السنة الكبيسة من ايام...
1,سيرابيوم\nالسيراپيوم مكان محفور فى صخور سقاره ...
2,ابريل\nابريل هو اليوم فى السنة الكبيسة من ايام...
3,قصر بعبدا\nقصر بعبدا هو مقر رئاسة الجمهورية ال...
4,سريان\nالسريان او الاشوريين هما الشعب الأرامى ...


In [ ]:
tweeets_data.head(5)

,Text
0,review
1,اكبر خطا ترتكبه ان تعامل الناس باخلاقك انت مش ...
2,دائما اكره اخر ليله في كل مكان .
3,يارب اللى يسرق تويتاتى يدخل النار .
4,الاسراف فى تناول القهوة يسبب الوفاه .


In [ ]:
# focusing

all_data = pd.concat([sports_data, Wikipedia_data, tweeets_data], ignore_index=True)

all_data.describe

<bound method NDFrame.describe of 0                    تحية عناصر   لأحمد حسن   الفيديو ..  
1        ق15: هجمة خطيرة للأهلي، أحمد فتحي يرسل عرضية ل...
2                     بعد جاحة دي انا هسحب كلامي تاني 😂😂✋ 
3                                ربع ساعه والتعادل السلبي 
4        فرصة في منتهي الخطورة من عبد الله و لكن الدفاع...
                               ...                        
51173    لنسعد ايامنا بالابتسامة بدلاً ان نملاها بالدموع. 
51174    مش هقولك غير ان نص الضحك اللي ضحكته فى حياتي ك...
51175    ربنا يوفقك ويسهلك وان شاء الله تعدي الفترة دي ...
51176    مبسوطة اوى عملت طريقة مكرونة جديدة و هى دلوقتى...
51177                              مفيش اجمل من حضن الام .
Name: Text, Length: 51178, dtype: object>

In [ ]:
print(all_data.shape)
all_data = all_data.dropna()
print("after nan values: ",all_data.shape)
all_data = all_data.drop_duplicates()
print("after duplicate values: ",all_data.shape)

(51178,)
after nan values:  (51178,)
after duplicate values:  (49424,)


## normalization

### Handle Inputes

In [ ]:

## done
_AR_DIGITS = {
    0: "صفر",
    1: "واحد",
    2: "اثنين",
    3: "تلتة",
    4: "أربعة",
    5: "خمسة",
    6: "ستة",
    7: "سبعة",
    8: "ثمانية",
    9: "تسعة",
    10: "عشرة",
    11: "احداشر",
    12: "اتناشر",
    13: "تلتاشر",
    14: "اربعتاشر",
    15: "خمستاشر",
    16: "ستاشر",
    17: "سبعتاشر",
    18: "ثمانتاشر",
    19: "تسعتاشر",
}

## done
_AR_TENS = {
    10: "عشرة",
    20: "عشرين",
    30: "تلاتين",
    40: "اربعين",
    50: "خمسين",
    60: "ستين",
    70: "سبعين",
    80: "ثمانين",
    90: "تسعين",
}

## done
_AR_HUNDREDS = {
    100: "مية",
    200: "ميتين",
    300: "تلتمية",
    400: "ربعمية",
    500: "خمسمية",
    600: "ستمية",
    700: "سبعمية",
    800: "تمنمية",
    900: "تسعمية"}

## done
_SCALES = {
    1000:"الف",
    2000:"الفين",
    3000:"تلتلاف",
    4000:"أربعنلاف",
    5000:"خمستلاف",
    6000:"ستلاف",
    7000:"سبعتلاف",
    8000:"تمنتلاف",
    9000:"تسعتلاف",
    10000:"عشرتلاف",
    11000:"احداشر الف",
    12000:"اتناشر الف",
    13000:"تلتاشر الف",
    14000:"اربعتاشر الف",
    15000:"خمستاشر الف",
    16000:"ستاشر الف",
    17000:"سبعتاشر الف",
    18000:"تمنتاشر الف",
    19000:"تسعتاشر الف",
    1000000:"مليون",
    2000000:"مليونين",
    1000000000:"مليار",
    2000000000:"مليارين"
    }

## done

_SCALES_ = [
    (1_000_000_000, "مليار", "مليارين", "ملايير"),
    (1_000_000, "مليون", "مليونين", "ملايين"),
    (1_000, "ألف", "ألفين", "آلاف")
]

## done
_ARABIC_TO_WESTERN = {
    "٠": "0",
    "١": "1",
    "٢": "2",
    "٣": "3",
    "٤": "4",
    "٥": "5",
    "٦": "6",
    "٧": "7",
    "٨": "8",
    "٩": "9"
}

## done
CURRENCY = {
    "EGP": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "ج": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "جنيه": ("جنيه", "جنيجنيههين", "جنيه", "قرش", "قرشين", "قروش"),
    "LE": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "£": ("جنيه", "جنيه", "جنيه", "قرش", "قرشين", "قروش"),
    "$": ("دولار", "دولارين", "دولار", "سنت", "سنتين", "سنتات"),
    "USD": ("دولار", "دولارين", "دولار", "سنت", "سنتين", "سنتات"),
    "€": ("يورو", "يوروهين", "يوروهات", "سنت", "سنتين", "سنتات"),
    "EUR": ("يورو", "يوروهين", "يوروهات", "سنت", "سنتين", "سنتات")
}

## done
MARKS = {
    "?": "علامة استفهام",
    "؟": "علامة استفهام",
    "!": "علامة تعجب"
}

## done
_AR_ORDINALS = {
    1: "الأول",
    2: "التاني",
    3: "التالت",
    4: "الرابع",
    5: "الخامس",
    6: "السادس",
    7: "السابع",
    8: "التامن",
    9: "التاسع",
    10: "العاشر"
}

## done
_AR_MONTHS = {
    1: "يناير",
    2: "فبراير",
    3: "مارس",
    4: "أبريل",
    5: "مايو",
    6: "يونيو",
    7: "يوليو",
    8: "أغسطس",
    9: "سبتمبر",
    10: "أكتوبر",
    11: "نوفمبر",
    12: "ديسمبر"
}

## done
ABBREV_MAP = {
        r"\bم\.": "متر",
        r"\bكم\.": "كيلومتر",
        r"\bكجم\.": "كيلوجرام",
        r"\bد\.": "دكتور",
        r"\bأ\.د\.": "أستاذ دكتور",
        r"\bش\.": "شارع",
        r"\bص\.ب\.": "صندوق بريد",
    }

# discovered we need some details to be keept for a better audio :
"""
# done
UNIFY_ARABIC_LETTERS = {
    "أ": "ا",
    "إ": "ا",
    "آ": "ا",
    "ة": "ه",
    "ى": "ي",
    "ؤ": "و",
    "ئ": "ي",
    "ـ": "",
    "ً": "",
    "ٌ": "",
    "ٍ": "",
    "َ": "",
    "ُ": "",
    "ِ": "",
    "ّ": "",
    "ْ": ""
}
"""
# done
UNIFY_ARABIC_LETTERS = {
    "ه":"ة",
    "ـ": "",
    "ً": "",
    "ٌ": "",
    "ٍ": "",
    "َ": "",
    "ُ": "",
    "ِ": "",
    "ّ": "",
    "ْ": ""
}

# done
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Symbols & pictographs
    "\U0001F680-\U0001F6FF"  # Transport & map symbols
    "\U0001F700-\U0001F77F"  # Alchemical symbols
    "\U0001F780-\U0001F7FF"  # Geometric shapes
    "\U0001F800-\U0001F8FF"  # Supplemental arrows
    "\U0001F900-\U0001F9FF"  # Supplemental symbols
    "\U0001FA00-\U0001FAFF"  # Chess / symbols
    "\U00002700-\U000027BF"  # Dingbats
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

DECIMAL_WORD = "نقطة"

PERCENT_WORD = "فى المية"


### helper funcitions

In [ ]:
# English Digits

def normalize_arabic_digits(text: str)-> str:
  """Convert Arabic digits to their equivalent English words."""
  return text.translate(str.maketrans(_ARABIC_TO_WESTERN))



def get_plural_form(n: int, singular: str, dual: str, plural: str) -> str:
    """Return appropriate Arabic plural form based on number."""
    if n == 1:
        return singular
    elif n == 2:
        return dual
    else:
        return plural

In [ ]:
# dealing with emojies

def remove_emojis(text: str) -> str:
    """
    Remove emojis from Arabic text.
    """
    return EMOJI_PATTERN.sub("", text)



In [ ]:

def int_to_egyptian_words(n: int) -> str:
    """Convert integer (>=0) into Egyptian Arabic-ish words with better scaling."""
    if n < 0:
        return "سالب " + int_to_egyptian_words(-n)
    if n < 20:
        return _AR_DIGITS[n]
    if n < 100:
        tens = (n // 10) * 10
        ones = n % 10
        if ones == 0:
            return _AR_TENS[tens]
        return f"{_AR_DIGITS[ones]} و{_AR_TENS[tens]}"
    if n < 1000:
        hundreds = (n // 100) * 100
        rest = n % 100
        if rest == 0:
            return _AR_HUNDREDS[hundreds]
        return f"{_AR_HUNDREDS[hundreds]} و{int_to_egyptian_words(rest)}"

    for scale_value, scale_singular, scale_dual, scale_plural in _SCALES_: ########################### changes for the new version
        if n >= scale_value:
            major = n // scale_value
            rest = n % scale_value

            # Better pluralization
            if major == 1:
                major_words = scale_singular
            elif major == 2:
                major_words = scale_dual
            elif major <= 10:
                major_words = f"{int_to_egyptian_words(major)} {scale_plural}"
            else:
                major_words = f"{int_to_egyptian_words(major)} {scale_singular}"

            if rest:
                major_words += f" و{int_to_egyptian_words(rest)}"
            return major_words

    return str(n)


In [ ]:

def num_token_to_words(token: str) -> str:
    """
    Convert token like:
      - "123" -> words
      - "12.5" or "12,5" -> words with DECIMAL_WORD
    Handles Arabic-Indic digits as well.
    """
    # Normalize Arabic digits first
    t = normalize_arabic_digits(token)
    t = t.replace("٬", "").replace(",", ".")

    if not re.fullmatch(r"-?\d+(\.\d+)?", t):
        return token

    neg = t.startswith("-")
    if neg:
        t = t[1:]

    if "." in t:
        a, b = t.split(".", 1)
        a_words = int_to_egyptian_words(int(a)) if a else "صفر"
        # Speak each digit in decimal part
        b_words = " ".join(_AR_DIGITS[int(ch)] for ch in b if ch.isdigit())
        out = f"{a_words} {DECIMAL_WORD} {b_words}".strip()
    else:
        out = int_to_egyptian_words(int(t))

    return ("سالب " + out) if neg else out


In [ ]:
# Specific pattern normalizers
# _SCALES

def normalize_percent(text: str) -> str:
    """Handle percentages including Arabic digits and symbols."""
    def repl(m):
        num = m.group("num")
        return f"{num_token_to_words(num)} {PERCENT_WORD}"

    # Handle both % and ٪ (Arabic percent)
    return re.sub(r"(?P<num>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\s*[%٪]+", repl, text)


def normalize_currency(text: str) -> str:
    """
    Enhanced currency handling with proper pluralization.
    Handles: "100 EGP", "100ج", "$12.50", "€30"
    """
    def sym_first(m):
        sym = m.group("sym")
        num = m.group("num")

        if sym not in CURRENCY:
            return m.group(0)

        major_sg, major_du, major_pl, minor_sg, minor_du, minor_pl = CURRENCY[sym]

        # Normalize and parse
        t = normalize_arabic_digits(num).replace(",", ".")

        if "." in t:
            a, b = t.split(".", 1)
            a_i = int(a) if a else 0
            b2 = (b + "00")[:2]
            b_i = int(b2)

            major_word = get_plural_form(a_i, major_sg, major_du, major_pl)

            if b_i == 0:
                return f"{int_to_egyptian_words(a_i)} {major_word}"

            minor_word = get_plural_form(b_i, minor_sg, minor_du, minor_pl)
            return f"{int_to_egyptian_words(a_i)} {major_word} و{int_to_egyptian_words(b_i)} {minor_word}"

        a_i = int(t)
        major_word = get_plural_form(a_i, major_sg, major_du, major_pl)
        return f"{num_token_to_words(num)} {major_word}"

    text = re.sub(r"(?P<sym>[$€£])\s*(?P<num>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)", sym_first, text)

    # Num-first pattern
    def num_first(m):
        num = m.group("num")
        cur = m.group("cur").strip()

        if cur not in CURRENCY:
            return m.group(0)

        major_sg, major_du, major_pl = CURRENCY[cur][:3]
        num_normalized = normalize_arabic_digits(num).replace(",", ".")

        if "." not in num_normalized:
            n = int(num_normalized)
            major_word = get_plural_form(n, major_sg, major_du, major_pl)
        else:
            major_word = major_sg

        return f"{num_token_to_words(num)} {major_word}"

    return re.sub(r"(?P<num>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\s*(?P<cur>EGP|USD|EUR|LE|جنيه|ج)", num_first, text)


def normalize_dates(text: str) -> str:
    """
    Handle date patterns:
    - 15/3/2024 -> "خمستاشر مارس ألفين وأربعة وعشرين"
    - 2024-03-15
    """
    def repl_slash(m):
        day = int(normalize_arabic_digits(m.group("day")))
        month = int(normalize_arabic_digits(m.group("month")))
        year = int(normalize_arabic_digits(m.group("year"))) if m.group("year") else None

        day_word = _AR_ORDINALS.get(day, int_to_egyptian_words(day))
        month_word = _AR_MONTHS.get(month, int_to_egyptian_words(month))

        if year:
            year_word = int_to_egyptian_words(year)
            return f"{day_word} {month_word} {year_word}"
        return f"{day_word} {month_word}"

    # DD/MM/YYYY or DD/MM
    text = re.sub(
        r"\b(?P<day>[\d٠-٩]{1,2})[/\-](?P<month>[\d٠-٩]{1,2})(?:[/\-](?P<year>[\d٠-٩]{2,4}))?\b",
        repl_slash,
        text
    )

    return text


def normalize_time(text: str) -> str:
    """Enhanced time handling with more natural Egyptian expressions."""
    def repl(m):
        hh = int(normalize_arabic_digits(m.group("hh")))
        mm = int(normalize_arabic_digits(m.group("mm")))

        h_words = int_to_egyptian_words(hh)

        if mm == 0:
            return f"{h_words} بالظبط"
        if mm == 15:
            return f"{h_words} وربع"
        if mm == 30:
            return f"{h_words} ونص"
        if mm == 45:
            next_h = (hh + 1) % 24
            return f"{int_to_egyptian_words(next_h)} إلا ربع"

        # For other minutes
        m_words = int_to_egyptian_words(mm)
        if mm == 1:
            return f"{h_words} ودقيقة"
        elif mm == 2:
            return f"{h_words} ودقيقتين"
        else:
            return f"{h_words} و{m_words} دقيقة"

    return re.sub(r"\b(?P<hh>[0-2]?[\d٠-٩]):(?P<mm>[0-5][\d٠-٩])\b", repl, text)


def normalize_ranges(text: str) -> str:
    """Handle numeric ranges with better context awareness."""
    def repl(m):
        a, b = m.group("a"), m.group("b")
        return f"من {num_token_to_words(a)} لحد {num_token_to_words(b)}"

    return re.sub(
        r"\b(?P<a>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\s*[-–—]\s*(?P<b>-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)\b",
        repl,
        text
    )


def normalize_phone_like(text: str) -> str:
    """
    Enhanced phone number handling.
    Recognizes Egyptian phone patterns (01X XXXX XXXX).
    """
    def repl(m):
        s = normalize_arabic_digits(m.group(0))
        s = re.sub(r"\D", "", s)

        # Don't process if too short (likely not a phone number)
        if len(s) < 7:
            return m.group(0)

        return " ".join(_AR_DIGITS[int(ch)] for ch in s)

    # Match Egyptian phone patterns and general long digit sequences
    return re.sub(r"(\+?[\d٠-٩][\d٠-٩\s().\-]{6,}[\d٠-٩])", repl, text)


def normalize_plain_numbers(text: str) -> str:
    """
    Convert remaining standalone numbers to words.
    Improved to avoid false positives.
    """
    def repl(m):
        return num_token_to_words(m.group(0))

    # Standalone numbers with Arabic digit support
    return re.sub(
        r"(?<![A-Za-zا-ي])(-?[\d٠-٩]+(?:[.,٫][\d٠-٩]+)?)(?![A-Za-zا-ي])",
        repl,
        text
    )


def normalize_abbreviations(text: str) -> str:
    """Expand common Egyptian abbreviations."""

    for pattern, replacement in ABBREV_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text

def normalize_unify_arabic_letters(text: str) -> str:
    """unify common Egyptian letters."""

    for pattern, replacement in UNIFY_ARABIC_LETTERS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text
"""
def normalize_marks(text: str) -> str:
    #unify common Egyptian letters.

    for pattern, replacement in MARKS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text
"""


'\ndef normalize_marks(text: str) -> str:\n    #unify common Egyptian letters.\n\n    for pattern, replacement in MARKS.items():\n        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)\n\n    return text\n'

In [ ]:

def normalize_text_for_tts_egyptian(text: str) -> str:
    """
    Enhanced pipeline with better ordering and new features.
    Order matters: specific patterns before general ones.
    """
    text = str(text)
    if not text or not text.strip():
        return ""

    text = text.strip()

    # Remove Emojis
    text = remove_emojis(text)

    # Normalize Arabic digits early
    text = normalize_arabic_digits(text)

    # Specific patterns first (most to least specific)
    text = normalize_dates(text)        # Before time (to avoid date slashes)
    text = normalize_time(text)
    text = normalize_percent(text)
    text = normalize_currency(text)
    text = normalize_ranges(text)       # Before plain numbers
    text = normalize_phone_like(text)   # Before plain numbers
    text = normalize_abbreviations(text)
    text = normalize_unify_arabic_letters(text)
    #text = normalize_marks(text)
    text = normalize_plain_numbers(text)  # Last for remaining numbers

    # Cleanup extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

### Test normalization

In [ ]:
# Data Normalization Final Stage :

In [ ]:
test_cases = [
    " بعد جاحة دي انا هسحب كلامي تاني 😂😂✋ ",
    "عايز 1 كتاب",
    "الفصل 1",
    "1 يناير",
    "15 يناير 2024",
    "الساعة 1:30",
    "السعر 100 جنيه",
    "تليفوني 01012345678",
    "في سنة 2024",
    "توقعاتك للشوط التاني؟",
    "بحيره ملاوي بحيره ملاوي هي واحده من بحيرات افريقيا العظمي تالت بحيرات افريقيا في المساحه والخامس عالميا من حيث الحجم تتقاسم دول شواطي البحيره مالاوي موزمبيق تنزانيا"
]

test_cases_answers = [
                      "بعد جاحه دي انا هسحب كلامي تاني",
                      "عايز واحد كتاب",
                      "الفصل الأول",
                       "أول يناير",
                       "خمستاشر يناير ألفين وأربعة وعشرين",
                       "الساعة واحدة ونص",
                       "السعر مية جنيه",
                       "تليفوني صفر-واحد-صفر...",
                       "تليفوني صفر-واحد-",
                      "في سنة ألفين وأربعة وعشرين",
                      "توقعاتك للشوط التاني علامة إستفهام"]

In [ ]:
normalized_test_cases = [normalize_text_for_tts_egyptian(text) for text in test_cases]
#normalize_text_for_tts_egyptian("15 يناير 2024")

In [ ]:
for text in normalized_test_cases:
  print(text)

بعد جاحة دي انا ةسحب كلامي تاني
عايز واحد كتاب
الفصل واحد
واحد يناير
خمستاشر يناير ألفين وأربعة وعشرين
الساعة واحد ونص
السعر مية جنية
تليفوني صفر واحد صفر واحد اثنين تلتة أربعة خمسة ستة سبعة ثمانية
في سنة ألفين وأربعة وعشرين
توقعاتك للشوط التاني؟
بحيرة ملاوي بحيرة ملاوي ةي واحدة من بحيرات افريقيا العظمي تالت بحيرات افريقيا في المساحة والخامس عالميا من حيث الحجم تتقاسم دول شواطي البحيرة مالاوي موزمبيق تنزانيا


In [ ]:
for text in test_cases_answers:
   print(text)

بعد جاحه دي انا هسحب كلامي تاني
عايز واحد كتاب
الفصل الأول
أول يناير
خمستاشر يناير ألفين وأربعة وعشرين
الساعة واحدة ونص
السعر مية جنيه
تليفوني صفر-واحد-صفر...
تليفوني صفر-واحد-
في سنة ألفين وأربعة وعشرين


### normalize our data:

In [ ]:
path = "/gdrive/MyDrive/Olimi AI /normalized_data.xlsx"

normalized = all_data.apply(normalize_text_for_tts_egyptian)

normalized.to_frame().to_excel(path, index=False)

In [ ]:
all_data.head(5)

,Text
0,تحية عناصر لأحمد حسن الفيديو ..
1,ق15: هجمة خطيرة للأهلي، أحمد فتحي يرسل عرضية ل...
2,بعد جاحة دي انا هسحب كلامي تاني 😂😂✋
3,ربع ساعه والتعادل السلبي
4,فرصة في منتهي الخطورة من عبد الله و لكن الدفاع...


In [ ]:
normalized.head(5)

,Text
0,تحيه عناصر لاحمد حسن الفيديو ..
1,ق1خمسة: هجمه خطيره للاهلي، احمد فتحي يرسل عرضي...
2,بعد جاحه دي انا هسحب كلامي تاني
3,ربع ساعه والتعادل السلبي
4,فرصه في منتهي الخطوره من عبد الله و لكن الدفاع...


### Run the saved normalized data :

In [ ]:
# Saved file for normalized_data :
path = "/content/drive/MyDrive/Olimi AI /normalized_data.xlsx"
normalized_data = pd.read_excel(path)
normalized_data.head(5)

,Text
0,تحيه عناصر لاحمد حسن الفيديو ..
1,ق1خمسة: هجمه خطيره للاهلي، احمد فتحي يرسل عرضي...
2,بعد جاحه دي انا هسحب كلامي تاني
3,ربع ساعه والتعادل السلبي
4,فرصه في منتهي الخطوره من عبد الله و لكن الدفاع...


In [ ]:
# Shuffle
normalized_data = normalized_data.sample(frac=1, random_state=42).reset_index(drop=True)


## NAMAA-Egyptian-TTS

In [ ]:
# save model :

"""

MODEL_DIR = "/gdrive/MyDrive/Olimi AI /model"
device = "cuda"
ckpt_dir = snapshot_download(
    repo_id="NAMAA-Space/NAMAA-Egyptian-TTS",
    repo_type="model",
    local_dir=MODEL_DIR,        # ← saves straight to Drive
    local_dir_use_symlinks=False  # ← important: real files, not symlinks
)

"""

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

##### ERROR FROM THE HOST : Packges not compatible with each other

In [ ]:
# load
from safetensors.torch import load_file as load_safetensors
from chatterbox import mtl_tts

MODEL_DIR = "/gdrive/MyDrive/Olimi AI /model/NAMAA-MODEL"
device = "cuda"

model = mtl_tts.ChatterboxMultilingualTTS.from_pretrained(device=device)
t3_state = load_safetensors(f"{MODEL_DIR}/t3_mtl23ls_v2.safetensors", device=device)
model.t3.load_state_dict(t3_state)
model.t3.to(device).eval()

print("Model loaded from Drive ✓")

## Edge-tts

In [ ]:
import edge_tts
import asyncio
from IPython.display import Audio, display

# example :

async def test():
    tts = edge_tts.Communicate(
        text="بحيره ملاوي بحيره ملاوي هي واحده من بحيرات افريقيا العظمي تالت بحيرات افريقيا في المساحه والخامس عالميا من حيث الحجم تتقاسم دول شواطي البحيره مالاوي موزمبيق تنزانيا",
        voice="ar-EG-SalmaNeural"
    )
    await tts.save("test.mp3")

await test()
display(Audio("test.mp3"))   # listen in Colab

In [ ]:
# importing

import edge_tts
import asyncio
import json
import os
from pathlib import Path

# config

# Config
VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]  # female + male
AUDIO_DIR = Path("/content/drive/MyDrive/Olimi AI /audio")
CHECKPOINT = "/content/drive/MyDrive/Olimi AI /prompts/checkpoint.json"
MANIFEST_OUT = "/content/drive/MyDrive/Olimi AI /prompts/manifest.jsonl"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Convert Series to Prompt List
# Replace 'text' with your actual column name from normalized_data.columns

TEXT_COLUMN = "Text"

prompts = [
    {
        "id": f"p_{i+1:04d}",        # p_0001, p_0002, ...
        "text": str(row[TEXT_COLUMN]).strip()
    }
    for i, row in normalized_data.iterrows()
    if str(row[TEXT_COLUMN]).strip() != ""   # skip empty rows
]

print(f"Clean prompts ready: {len(prompts)}")
print(prompts[0])   # verify first one looks right

Clean prompts ready: 49424
{'id': 'p_0001', 'text': 'صباح الاختبارات .'}


In [ ]:
# Save as JSONL

import json

PROMPT_MANIFEST = "/content/drive/MyDrive/Olimi AI /prompts/manifest.jsonl"

with open(PROMPT_MANIFEST, "w", encoding="utf-8") as f:
    for p in prompts:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Saved {len(prompts)} prompts to manifest")

Saved 49424 prompts to manifest


In [ ]:
# Load checkpoint
done = set(json.load(open(CHECKPOINT))) if os.path.exists(CHECKPOINT) else set()

# one by one
async def synthesize_one(prompt: dict, voice: str) -> dict:
    out_path = AUDIO_DIR / f"{prompt['id']}.mp3"
    try:
        tts = edge_tts.Communicate(text=prompt['text'], voice=voice)
        await tts.save(str(out_path))
        return {**prompt, "audio_path": str(out_path),
                "voice": voice, "status": "success"}
    except Exception as e:
        return {**prompt, "status": "failed", "error": str(e)}


# handle onley 100 example :
prompts = prompts[:100]

async def run_synthesis(prompts: list):
    manifest = []
    pending = [p for p in prompts if p['id'] not in done]
    print(f"Total: {len(prompts)} | Done: {len(done)} | Pending: {len(pending)}")

    for i, prompt in enumerate(pending):
        voice = VOICES[i % len(VOICES)]   # alternate voices
        result = await synthesize_one(prompt, voice)

        if result['status'] == 'success':
            done.add(prompt['id'])
            manifest.append(result)

            # Save checkpoint after every sample
            json.dump(list(done), open(CHECKPOINT, "w"))

            # Append to manifest
            with open(MANIFEST_OUT, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            print(f"✓ [{i+1}/{len(pending)}] {prompt['id']}")
        else:
            print(f"✗ {prompt['id']}: {result['error']}")

        await asyncio.sleep(0.3)   # avoid rate limiting


In [ ]:
# Everything together — read, shuffle, convert, synthesize
await run_synthesis(prompts)

Total: 100 | Done: 0 | Pending: 100
✓ [1/100] p_0001
✓ [2/100] p_0002
✓ [3/100] p_0003
✓ [4/100] p_0004
✓ [5/100] p_0005
✓ [6/100] p_0006
✓ [7/100] p_0007
✓ [8/100] p_0008
✓ [9/100] p_0009
✓ [10/100] p_0010
✓ [11/100] p_0011
✓ [12/100] p_0012
✓ [13/100] p_0013
✓ [14/100] p_0014
✓ [15/100] p_0015
✓ [16/100] p_0016
✓ [17/100] p_0017
✓ [18/100] p_0018
✓ [19/100] p_0019
✓ [20/100] p_0020
✓ [21/100] p_0021
✓ [22/100] p_0022
✓ [23/100] p_0023
✓ [24/100] p_0024
✓ [25/100] p_0025
✓ [26/100] p_0026
✓ [27/100] p_0027
✓ [28/100] p_0028
✓ [29/100] p_0029
✓ [30/100] p_0030
✓ [31/100] p_0031
✓ [32/100] p_0032
✓ [33/100] p_0033
✓ [34/100] p_0034
✓ [35/100] p_0035
✓ [36/100] p_0036
✓ [37/100] p_0037
✓ [38/100] p_0038
✓ [39/100] p_0039
✓ [40/100] p_0040
✓ [41/100] p_0041
✓ [42/100] p_0042
✓ [43/100] p_0043
✓ [44/100] p_0044
✓ [45/100] p_0045
✓ [46/100] p_0046
✓ [47/100] p_0047
✓ [48/100] p_0048
✓ [49/100] p_0049
✓ [50/100] p_0050
✓ [51/100] p_0051
✓ [52/100] p_0052
✓ [53/100] p_0053
✓ [54/100] p_0054
✓